# Notebook 27 — multi-stream open-set gate

**Runtime: T4 GPU.** Press **Run all**. The Notebook 26 temporal exercise
classifier is frozen. Only the binary gate changes: separate joint, bone, and
velocity TCN streams are trained, then validation selects a CrossMax-inspired
fusion and threshold under an explicit 85% unknown-rejection constraint. The
1,971-window test is read only after selection. The final download is
`calorie_v6f_multistream_open_set_gate_report.zip`.


In [ ]:
%pip install -q scikit-learn seaborn pandas==2.2.3


In [ ]:
from pathlib import Path
import hashlib
import json
import random
import shutil
import urllib.request
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from google.colab import files
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    confusion_matrix, f1_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from torch import nn
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler

SEED = 42
WINDOW = 64
STRIDE = 32
BATCH_SIZE = 64
MAX_EPOCHS = 40
PATIENCE = 8
MIN_VALIDATION_UNKNOWN_REJECTION = 0.85
GROUP_TEMPERATURE = 0.5  # frozen from V6-A.1
AUGMENTATION_CONFIG = {
    "horizontal_flip_probability": 0.5,
    "rotation_probability": 0.7,
    "max_rotation_degrees": 7.0,
    "speed_probability": 0.5,
    "speed_range": [0.85, 1.15],
    "frame_drop_probability": 0.35,
    "frame_drop_fraction_range": [0.05, 0.12],
    "coordinate_noise_probability": 0.8,
    "coordinate_noise_std": 0.008,
}
RETIRED_CLASSES = {"plank"}
NEW_TRAIN_GROUPS = {"youtube:Fk1wnNX4wQ8", "youtube:zxqCV_eQHZY"}
UNREVIEWED_GROUPS = {
    "youtube:oLsonHQJOUc",
    "youtube:c4TomZjzobY",
    "youtube:W2Bm9dfQjbk",
}


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


seed_everything(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
print("Experiment: multi-stream open-set gate with frozen temporal classifier")
assert DEVICE.type == "cuda", "Select a T4 runtime before training."


## Download pinned evidence and freeze the temporal classifier

All inputs come from commit `9874773`. Notebook 26's exercise state is loaded
and frozen; only three binary gate streams train.


In [ ]:
PINNED_COMMIT = "9874773e4681464bb00f5f0fb4dc3459caf75d6c"
REPOSITORY_RAW = (
    "https://raw.githubusercontent.com/bariscelikk1/cALorie/"
    f"{PINNED_COMMIT}/ml/artifacts"
)
LANDMARK_ARCHIVES = [
    ("v1/landmarks", "calorie_workoutfitness_landmarks_v1.zip"),
    ("v1/landmarks", "calorie_mmfit_landmarks_v1.zip"),
    ("v1/landmarks", "calorie_mmfit_unknown_v1.zip"),
    ("v2/landmarks", "calorie_youtube_mixed_landmarks_v3.zip"),
    ("v3/landmarks", "calorie_youtube_additional_landmarks_v1.zip"),
    ("v3/landmarks", "calorie_youtube_training_landmarks_v4.zip"),
    ("v5/landmarks", "calorie_workoutfitness_gate_hardneg_v1.zip"),
    ("youtube_long_v2/reviewed_clips", "calorie_youtube_long_workouts_v2_reviewed_clips.zip"),
]
EXPECTED_HASHES = {
    "calorie_youtube_long_workouts_v2_reviewed_clips.zip": "84fb55b981820af702a94f0c0070960b24fdd1acb617097f1838c782cd942284",
    "hard_positive_windows.csv": "5a4b73c04ee85659570e7931ea0e84b35c0abeb16fd3f08238e629f6eca091c5",
    "split_assignments.csv": "8c013695fe095ceccc4ff65722d2a26e5a309674f5b877b840b71ba73dc085d6",
    "baseline_metrics.json": "baee2ea099402ca25b0e2bc322809dfa7a0f8d1e9969204792cd5d60bfb96b2b",
    "baseline_test_predictions.csv": "b34b6cada90355682ef7aa225e3d18dad0960dd79ce32856ebb524356734326c",
    "data_only_metrics.json": "cdc3d944063d12226a2ceaab6be9afd4e4cc23b6031dddc52a6ad86bb43c0670",
    "data_only_test_predictions.csv": "b379c30e563ea626d1f411c3ac372280820db03e7bd86dded647bc90a14adc65",
    "temporal_checkpoint.pt": "c30ecd625d77f31380a503f881eff5c540b8a7cdbbb71a39276c33b908947b92",
    "temporal_metrics.json": "27d15503aae2c23447294612569fcc3f96b08735ef3576f794c7db1d14824dcb",
    "temporal_test_predictions.csv": "7cab864936c31515ef65b2e55ea7a79ed282d2377468be7c68ca3b8120e76716",
}


def sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def download(url, destination, expected=None):
    if destination.exists() and (expected is None or sha256(destination) == expected):
        print("Cached:", destination.name)
        return
    temporary = destination.with_suffix(destination.suffix + ".part")
    temporary.unlink(missing_ok=True)
    request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(request, timeout=300) as response, temporary.open("wb") as output:
        shutil.copyfileobj(response, output)
    if expected is not None:
        actual = sha256(temporary)
        assert actual == expected, f"SHA-256 mismatch for {destination.name}: {actual}"
    temporary.replace(destination)
    print(f"{destination.name}: {destination.stat().st_size / 1e6:.1f} MB")


zip_paths = []
for relative_root, name in LANDMARK_ARCHIVES:
    destination = Path("/content") / name
    download(
        f"{REPOSITORY_RAW}/{relative_root}/{name}", destination,
        EXPECTED_HASHES.get(name),
    )
    with zipfile.ZipFile(destination) as archive:
        assert archive.testzip() is None, f"Corrupt ZIP: {name}"
    zip_paths.append(destination)

HARD_POSITIVE_PATH = Path("/content/hard_positive_windows.csv")
download(
    f"{REPOSITORY_RAW}/v5/hard_positive_mining/hard_positive_windows.csv",
    HARD_POSITIVE_PATH, EXPECTED_HASHES["hard_positive_windows.csv"],
)
hard_positive_frame = pd.read_csv(HARD_POSITIVE_PATH)
assert hard_positive_frame.window_key.is_unique

V6_SPLIT_PATH = Path("/content/split_assignments.csv")
download(
    f"{REPOSITORY_RAW}/v6a1/model/split_assignments.csv",
    V6_SPLIT_PATH, EXPECTED_HASHES["split_assignments.csv"],
)
BASELINE_METRICS_PATH = Path("/content/baseline_metrics.json")
download(
    f"{REPOSITORY_RAW}/v6c/model/metrics.json",
    BASELINE_METRICS_PATH, EXPECTED_HASHES["baseline_metrics.json"],
)
BASELINE_PREDICTIONS_PATH = Path("/content/baseline_test_predictions.csv")
download(
    f"{REPOSITORY_RAW}/v6c/model/test_window_predictions.csv",
    BASELINE_PREDICTIONS_PATH, EXPECTED_HASHES["baseline_test_predictions.csv"],
)
baseline_metrics = json.loads(BASELINE_METRICS_PATH.read_text())
baseline_predictions = pd.read_csv(BASELINE_PREDICTIONS_PATH)
DATA_ONLY_METRICS_PATH = Path("/content/data_only_metrics.json")
download(f"{REPOSITORY_RAW}/v6c_longv2_data_retrain/model/metrics.json", DATA_ONLY_METRICS_PATH, EXPECTED_HASHES["data_only_metrics.json"])
DATA_ONLY_PREDICTIONS_PATH = Path("/content/data_only_test_predictions.csv")
download(f"{REPOSITORY_RAW}/v6c_longv2_data_retrain/model/test_window_predictions.csv", DATA_ONLY_PREDICTIONS_PATH, EXPECTED_HASHES["data_only_test_predictions.csv"])
data_only_metrics = json.loads(DATA_ONLY_METRICS_PATH.read_text())
data_only_predictions = pd.read_csv(DATA_ONLY_PREDICTIONS_PATH)
TEMPORAL_CHECKPOINT_PATH = Path("/content/temporal_checkpoint.pt")
download(f"{REPOSITORY_RAW}/v6e_complete_action_temporal/model/two_stage_tcn_v6e_temporal.pt", TEMPORAL_CHECKPOINT_PATH, EXPECTED_HASHES["temporal_checkpoint.pt"])
TEMPORAL_METRICS_PATH = Path("/content/temporal_metrics.json")
download(f"{REPOSITORY_RAW}/v6e_complete_action_temporal/model/metrics.json", TEMPORAL_METRICS_PATH, EXPECTED_HASHES["temporal_metrics.json"])
TEMPORAL_PREDICTIONS_PATH = Path("/content/temporal_test_predictions.csv")
download(f"{REPOSITORY_RAW}/v6e_complete_action_temporal/model/test_window_predictions.csv", TEMPORAL_PREDICTIONS_PATH, EXPECTED_HASHES["temporal_test_predictions.csv"])
temporal_metrics = json.loads(TEMPORAL_METRICS_PATH.read_text())
temporal_predictions = pd.read_csv(TEMPORAL_PREDICTIONS_PATH)

SOURCE_ROOT = Path("/content/v6c_longv2_sources")
if SOURCE_ROOT.exists():
    shutil.rmtree(SOURCE_ROOT)
SOURCE_ROOT.mkdir()
for index, archive_path in enumerate(zip_paths):
    destination = SOURCE_ROOT / f"source_{index}"
    destination.mkdir()
    with zipfile.ZipFile(archive_path) as archive:
        archive.extractall(destination)

frames = []
for manifest_path in SOURCE_ROOT.rglob("manifest.csv"):
    frame = pd.read_csv(manifest_path).fillna("")
    frame["npz_path"] = frame["output_path"].map(
        lambda value: str(manifest_path.parent / value)
    )
    frames.append(frame)
assert frames, "No manifest.csv was found."
manifest = pd.concat(frames, ignore_index=True)
manifest = manifest[
    (manifest.status == "ok")
    & manifest.npz_path.map(lambda value: Path(value).exists())
].copy()
if "group_id" not in manifest or manifest.group_id.eq("").any():
    manifest["group_id"] = manifest.apply(
        lambda row: f"{row['source_dataset']}:{row['clip_id']}", axis=1
    )
assert NEW_TRAIN_GROUPS <= set(manifest.group_id)
print("Valid clips:", len(manifest), "Groups:", manifest.group_id.nunique())
display(manifest.canonical_class.value_counts().sort_index().rename("clips"))


## Preserve fixed windows and add train-only phase cycles

The old split and all evaluation windows remain unchanged. Supported training
clips receive at most six autocorrelation-period candidates. Each candidate is
resampled by cumulative joint motion, so slow and fast phases receive comparable
coverage without changing validation or test evidence.


In [ ]:
def resample_sequence(sequence, source_fps, target_fps=15):
    if not source_fps or abs(float(source_fps) - target_fps) < 0.1:
        return sequence
    duration = (len(sequence) - 1) / float(source_fps)
    target_length = max(2, int(round(duration * target_fps)) + 1)
    old_x = np.linspace(0, duration, len(sequence))
    new_x = np.linspace(0, duration, target_length)
    flat = sequence.reshape(len(sequence), -1)
    sampled = np.stack([
        np.interp(new_x, old_x, flat[:, index])
        for index in range(flat.shape[1])
    ], axis=1)
    return sampled.reshape(target_length, 17, 4).astype(np.float32)


windows, original_labels, groups = [], [], []
clip_ids, sources, starts = [], [], []
for row in manifest.itertuples(index=False):
    with np.load(row.npz_path, allow_pickle=False) as item:
        sequence = item["landmarks"].astype(np.float32)
    if sequence.ndim != 3 or sequence.shape[1:] != (17, 4):
        print("Skipping unexpected shape:", row.npz_path, sequence.shape)
        continue
    sequence = resample_sequence(sequence, getattr(row, "source_fps", 15) or 15)
    clip_starts = list(range(0, max(1, len(sequence) - WINDOW + 1), STRIDE))
    if len(sequence) > WINDOW and clip_starts[-1] != len(sequence) - WINDOW:
        clip_starts.append(len(sequence) - WINDOW)
    for start in clip_starts:
        window = sequence[start:start + WINDOW]
        if len(window) < WINDOW:
            window = np.concatenate([
                window, np.repeat(window[-1:], WINDOW - len(window), axis=0)
            ])
        windows.append(window)
        original_labels.append(row.canonical_class)
        groups.append(row.group_id)
        clip_ids.append(row.clip_id)
        sources.append(row.source_dataset)
        starts.append(start)

X = np.asarray(windows, dtype=np.float32)
original_labels = np.asarray(original_labels)
groups = np.asarray(groups)
clip_ids = np.asarray(clip_ids)
sources = np.asarray(sources)
starts = np.asarray(starts, dtype=np.int64)
window_keys = np.asarray([
    f"{source}|{group}|{clip}|{start}"
    for source, group, clip, start in zip(sources, groups, clip_ids, starts)
])

# Reuse the exact group assignment exported by V6-A.1.
frozen_split = pd.read_csv(V6_SPLIT_PATH)
assert frozen_split.groupby("group_id")["split"].nunique().max() == 1
group_to_split = (
    frozen_split[["group_id", "split"]]
    .drop_duplicates("group_id")
    .set_index("group_id")["split"]
    .to_dict()
)
old_evidence_mask = np.asarray([group in group_to_split for group in groups])
assigned_split = np.asarray([group_to_split.get(group, "") for group in groups])
new_train_mask = np.isin(groups, list(NEW_TRAIN_GROUPS))
assigned_split[new_train_mask] = "train"
missing_groups = set(groups[assigned_split == ""])
assert missing_groups <= UNREVIEWED_GROUPS, (
    "Unexpected unresolved groups: " + str(sorted(missing_groups))
)
assert set(groups[new_train_mask]) == NEW_TRAIN_GROUPS
train_idx = np.flatnonzero(assigned_split == "train")
val_idx = np.flatnonzero(assigned_split == "validation")
test_idx = np.flatnonzero(assigned_split == "test")

labels = np.where(np.isin(original_labels, list(RETIRED_CLASSES)), "unknown", original_labels)
ALL_CLASSES = sorted(np.unique(labels).tolist())
SUPPORTED_CLASSES = [name for name in ALL_CLASSES if name != "unknown"]
assert len(SUPPORTED_CLASSES) == 14, SUPPORTED_CLASSES
all_to_id = {name: index for index, name in enumerate(ALL_CLASSES)}
supported_to_id = {name: index for index, name in enumerate(SUPPORTED_CLASSES)}
y_all = np.asarray([all_to_id[name] for name in labels], dtype=np.int64)
y_gate = (labels != "unknown").astype(np.int64)
y_exercise = np.asarray([supported_to_id.get(name, -1) for name in labels], dtype=np.int64)

assert not (set(groups[train_idx]) & set(groups[val_idx]))
assert not (set(groups[train_idx]) & set(groups[test_idx]))
assert not (set(groups[val_idx]) & set(groups[test_idx]))
assert not (set(groups[train_idx]) & UNREVIEWED_GROUPS)
assert set(window_keys[old_evidence_mask]) == set(frozen_split.window_key), (
    "Reviewed window evidence changed since V6-A.1."
)
assert set(window_keys[test_idx]) == set(baseline_predictions.window_key)
assert len(test_idx) == 1971, len(test_idx)
assert set(window_keys[test_idx]) == set(data_only_predictions.window_key)
assert set(window_keys[test_idx]) == set(temporal_predictions.window_key)
baseline_truth_by_key = dict(zip(baseline_predictions.window_key, baseline_predictions.true_class))
assert all(baseline_truth_by_key[key] == truth for key, truth in zip(window_keys[test_idx], labels[test_idx])), "Frozen test labels changed relative to V6-C."
assert not set(groups[val_idx]) & NEW_TRAIN_GROUPS
assert not set(groups[test_idx]) & NEW_TRAIN_GROUPS
print("Excluded unresolved groups:", sorted(missing_groups))
print("New train-only groups:", sorted(NEW_TRAIN_GROUPS))

print("Windows:", X.shape)
print("Frozen groups:", len(group_to_split))
print("Active supported classes:", SUPPORTED_CLASSES)
print("Retired plank windows mapped to unknown:", int((original_labels == "plank").sum()))
for name, indices in [("train", train_idx), ("validation", val_idx), ("test", test_idx)]:
    print(name, "windows=", len(indices), "groups=", len(np.unique(groups[indices])))
    display(pd.Series(labels[indices]).value_counts().sort_index().rename("windows"))


new_indices = np.flatnonzero(new_train_mask)
data_delta_audit = pd.DataFrame({
    "canonical_class": labels[new_indices],
    "group_id": groups[new_indices],
    "window_key": window_keys[new_indices],
}).groupby(["canonical_class", "group_id"]).agg(
    windows=("window_key", "count")
).reset_index()
class_creator_support_matrix = pd.crosstab(
    pd.Series(labels[train_idx], name="canonical_class"),
    pd.Series(groups[train_idx], name="group_id"),
)
print("Added training windows:", len(new_indices))
display(data_delta_audit)
display(class_creator_support_matrix)


# Add train-only complete-cycle candidates while retaining every fixed window.
def estimate_period(sequence, minimum=12, maximum=90):
    xy = sequence[:, :, :2]
    energy = np.r_[0.0, np.linalg.norm(np.diff(xy, axis=0), axis=2).mean(axis=1)]
    energy = np.convolve(energy, np.ones(5) / 5.0, mode="same")
    centered = energy - energy.mean()
    upper = min(maximum, len(sequence) // 2)
    if upper <= minimum:
        return max(minimum, len(sequence))
    scores = []
    for lag in range(minimum, upper + 1):
        left, right = centered[:-lag], centered[lag:]
        denom = np.linalg.norm(left) * np.linalg.norm(right)
        scores.append(float(np.dot(left, right) / max(denom, 1e-8)))
    return minimum + int(np.argmax(scores))


def phase_resample(segment, target_length=WINDOW):
    xy = segment[:, :, :2]
    step_motion = np.r_[0.0, np.linalg.norm(np.diff(xy, axis=0), axis=2).mean(axis=1)]
    progress = np.cumsum(step_motion + 1e-5)
    progress = (progress - progress[0]) / max(progress[-1] - progress[0], 1e-8)
    targets = np.linspace(0.0, 1.0, target_length)
    flat = segment.reshape(len(segment), -1)
    sampled = np.stack([np.interp(targets, progress, flat[:, column]) for column in range(flat.shape[1])], axis=1)
    return sampled.reshape(target_length, 17, 4).astype(np.float32)


temporal_windows, temporal_labels, temporal_groups = [], [], []
temporal_clip_ids, temporal_sources, temporal_starts, temporal_rows = [], [], [], []
for row in manifest.itertuples(index=False):
    canonical = "unknown" if row.canonical_class in RETIRED_CLASSES else row.canonical_class
    if (group_to_split.get(row.group_id) != "train" and row.group_id not in NEW_TRAIN_GROUPS) or canonical == "unknown":
        continue
    with np.load(row.npz_path, allow_pickle=False) as item:
        sequence = item["landmarks"].astype(np.float32)
    sequence = resample_sequence(sequence, getattr(row, "source_fps", 15) or 15)
    if len(sequence) < 24:
        continue
    period = estimate_period(sequence)
    candidate_starts = list(range(0, max(1, len(sequence) - period + 1), period))
    if len(candidate_starts) > 6:
        candidate_starts = [candidate_starts[i] for i in np.linspace(0, len(candidate_starts) - 1, 6, dtype=int)]
    for cycle_index, start in enumerate(candidate_starts):
        end = min(len(sequence), start + period)
        if end - start < 12:
            continue
        temporal_windows.append(phase_resample(sequence[start:end]))
        temporal_labels.append(canonical); temporal_groups.append(row.group_id)
        temporal_clip_ids.append(row.clip_id); temporal_sources.append(row.source_dataset); temporal_starts.append(start)
        temporal_rows.append({"window_key": f"phase64|{row.source_dataset}|{row.group_id}|{row.clip_id}|{cycle_index}", "clip_id": row.clip_id, "group_id": row.group_id, "canonical_class": canonical, "source_start_frame": start, "source_end_frame": end, "estimated_period_frames": period, "source_frames": end - start})

assert temporal_windows, "No complete-action candidates were built."
temporal_frame = pd.DataFrame(temporal_rows)
base_count = len(X)
X = np.concatenate([X, np.asarray(temporal_windows, dtype=np.float32)])
original_labels = np.concatenate([original_labels, np.asarray(temporal_labels)])
labels = np.concatenate([labels, np.asarray(temporal_labels)])
groups = np.concatenate([groups, np.asarray(temporal_groups)])
clip_ids = np.concatenate([clip_ids, np.asarray(temporal_clip_ids)])
sources = np.concatenate([sources, np.asarray(temporal_sources)])
starts = np.concatenate([starts, np.asarray(temporal_starts, dtype=np.int64)])
window_keys = np.concatenate([window_keys, temporal_frame.window_key.to_numpy()])
assigned_split = np.concatenate([assigned_split, np.repeat("train", len(temporal_windows))])
y_all = np.asarray([all_to_id[name] for name in labels], dtype=np.int64)
y_gate = (labels != "unknown").astype(np.int64)
y_exercise = np.asarray([supported_to_id.get(name, -1) for name in labels], dtype=np.int64)
train_idx = np.flatnonzero(assigned_split == "train")
val_idx = np.flatnonzero(assigned_split == "validation")
test_idx = np.flatnonzero(assigned_split == "test")
temporal_mask = np.arange(len(X)) >= base_count
assert temporal_mask[val_idx].sum() == 0 and temporal_mask[test_idx].sum() == 0
assert set(window_keys[test_idx]) == set(data_only_predictions.window_key)
assert set(window_keys[test_idx]) == set(temporal_predictions.window_key)
print("Added phase-resampled training cycles:", int(temporal_mask.sum()))
display(temporal_frame.groupby("canonical_class").agg(cycles=("window_key", "count"), clips=("clip_id", "nunique"), median_period=("estimated_period_frames", "median")).round(2))


## Tempered hierarchical sampling and hard-positive weighting

Sampling first assigns equal probability to each class. Inside a class, each
source group receives probability proportional to `window_count ** 0.5`. This is
the midpoint between failed V6-A equal-group sampling (`** 0`) and raw window
sampling (`** 1`): long videos are restrained without making a one-window group
as influential as a large group. Notebook 13 hard-positive modifiers are
normalized inside the same class/group, so they change which windows are seen
without changing that group's total probability.

For the gate, “class” means supported or unknown. For the exercise model it
means one of the 14 active movements. `WeightedRandomSampler` then draws the
requested class -> group -> window distribution.


In [ ]:
hard_weight_by_key = dict(zip(
    hard_positive_frame.window_key,
    hard_positive_frame.recommended_sample_weight,
))
hard_positive_weights = np.asarray([
    float(hard_weight_by_key.get(key, 1.0)) for key in window_keys
], dtype=np.float64)
# Retired plank is now unknown and cannot receive a supported hard-positive boost.
hard_positive_weights[labels == "unknown"] = 1.0
matched_training_keys = (
    set(window_keys[train_idx][labels[train_idx] != "unknown"])
    & set(hard_weight_by_key)
)
assert not (set(window_keys[val_idx]) & set(hard_weight_by_key)), (
    "Notebook 13 weights unexpectedly intersect validation."
)
assert not (set(window_keys[test_idx]) & set(hard_weight_by_key)), (
    "Notebook 13 weights unexpectedly intersect test."
)
print("Notebook 13 keys:", len(hard_weight_by_key))
print("Matched active training keys:", len(matched_training_keys))
print("Upweighted active training windows:", int((hard_positive_weights[train_idx] > 1).sum()))


def hierarchical_window_weights(indices, hierarchy_labels, extra_weights=None):
    table = pd.DataFrame({
        "index": indices,
        "hierarchy_label": hierarchy_labels[indices],
        "group": groups[indices],
    })
    class_count = table["hierarchy_label"].nunique()
    group_sizes = table.groupby(
        ["hierarchy_label", "group"]
    )["index"].transform("count").astype(float)
    table["group_mass"] = group_sizes.pow(GROUP_TEMPERATURE)
    group_table = table[["hierarchy_label", "group"]].copy()
    group_table["group_mass"] = table["group_mass"]
    class_group_mass = (
        group_table.drop_duplicates(["hierarchy_label", "group"])
        .groupby("hierarchy_label")["group_mass"].sum()
    )
    table["extra_weight"] = (
        1.0 if extra_weights is None else extra_weights[table["index"].to_numpy()]
    )
    table["pair_extra_total"] = table.groupby(
        ["hierarchy_label", "group"]
    )["extra_weight"].transform("sum")
    weights = []
    for row in table.itertuples(index=False):
        weight = (
            1.0 / class_count
            * row.group_mass / class_group_mass.loc[row.hierarchy_label]
            * row.extra_weight / row.pair_extra_total
        )
        weights.append(weight)
    weights = np.asarray(weights, dtype=np.float64)
    return weights / weights.mean()


# Use strings here so the audit table remains human-readable.
gate_hierarchy = np.where(y_gate == 1, "supported", "unknown")
gate_train_weights = hierarchical_window_weights(
    train_idx, gate_hierarchy, hard_positive_weights
)
exercise_train_idx = train_idx[y_exercise[train_idx] >= 0]
exercise_val_idx = val_idx[y_exercise[val_idx] >= 0]
exercise_test_idx = test_idx[y_exercise[test_idx] >= 0]
exercise_train_weights = hierarchical_window_weights(
    exercise_train_idx, labels
)

sampler_audit = pd.DataFrame({
    "window_key": window_keys[train_idx],
    "canonical_class": labels[train_idx],
    "original_class": original_labels[train_idx],
    "group_id": groups[train_idx],
    "gate_target": gate_hierarchy[train_idx],
    "hard_positive_weight": hard_positive_weights[train_idx],
    "gate_sampler_weight": gate_train_weights,
})
exercise_weight_map = dict(zip(
    window_keys[exercise_train_idx], exercise_train_weights
))
sampler_audit["exercise_sampler_weight"] = sampler_audit.window_key.map(
    exercise_weight_map
).fillna(0.0)

display(sampler_audit.groupby("gate_target").agg(
    windows=("window_key", "count"),
    groups=("group_id", "nunique"),
    total_sampler_weight=("gate_sampler_weight", "sum"),
).round(3))
display(sampler_audit[sampler_audit.canonical_class != "unknown"].groupby(
    "canonical_class"
).agg(
    windows=("window_key", "count"),
    groups=("group_id", "nunique"),
    total_exercise_weight=("exercise_sampler_weight", "sum"),
).round(3))


In [ ]:
LEFT_RIGHT_PAIRS = [
    (1, 2), (3, 4), (5, 6), (7, 8),
    (9, 10), (11, 12), (13, 14), (15, 16),
]


def augment_pose_window(sample, force_all=False):
    sample = sample.copy()
    config = AUGMENTATION_CONFIG

    if force_all or np.random.random() < config["horizontal_flip_probability"]:
        sample[:, :, 0] *= -1.0
        for left, right in LEFT_RIGHT_PAIRS:
            sample[:, [left, right], :] = sample[:, [right, left], :]

    if force_all or np.random.random() < config["rotation_probability"]:
        degrees = np.random.uniform(
            -config["max_rotation_degrees"], config["max_rotation_degrees"]
        )
        radians = np.deg2rad(degrees)
        cosine, sine = np.cos(radians), np.sin(radians)
        x = sample[:, :, 0].copy()
        y = sample[:, :, 1].copy()
        sample[:, :, 0] = cosine * x - sine * y
        sample[:, :, 1] = sine * x + cosine * y

    if force_all or np.random.random() < config["speed_probability"]:
        speed = np.random.uniform(*config["speed_range"])
        center = (WINDOW - 1) / 2.0
        positions = (np.arange(WINDOW) - center) * speed + center
        flat = sample.reshape(WINDOW, -1)
        flat = np.stack([
            np.interp(positions, np.arange(WINDOW), flat[:, column])
            for column in range(flat.shape[1])
        ], axis=1)
        sample = flat.reshape(WINDOW, 17, 4).astype(np.float32)

    if force_all or np.random.random() < config["frame_drop_probability"]:
        fraction = np.random.uniform(*config["frame_drop_fraction_range"])
        count = max(1, int(round(WINDOW * fraction)))
        dropped = np.random.choice(np.arange(1, WINDOW), size=count, replace=False)
        sample[dropped] = sample[dropped - 1]

    if force_all or np.random.random() < config["coordinate_noise_probability"]:
        sample[:, :, :2] += np.random.normal(
            0, config["coordinate_noise_std"], sample[:, :, :2].shape
        ).astype(np.float32)
    return sample.astype(np.float32)



BONE_EDGES = [
    (5, 6), (5, 7), (7, 9), (6, 8), (8, 10),
    (5, 11), (6, 12), (11, 12),
    (11, 13), (13, 15), (12, 14), (14, 16),
]
ANGLE_TRIPLETS = [
    (5, 7, 9), (6, 8, 10),       # elbows
    (7, 5, 11), (8, 6, 12),      # shoulders
    (5, 11, 13), (6, 12, 14),    # hips
    (11, 13, 15), (12, 14, 16),  # knees
]
FEATURE_CONFIG = {
    "coordinates_xy": 34,
    "velocity_xy": 34,
    "bone_vectors_xy": 24,
    "joint_angles": 8,
}
FEATURE_DIM = sum(FEATURE_CONFIG.values())


def pose_to_features(sample):
    """Convert one [time, 17, 4] normalized pose window to [time, 100]."""
    xy = sample[:, :, :2].astype(np.float32)

    velocity = np.zeros_like(xy)
    velocity[1:] = xy[1:] - xy[:-1]

    bones = np.stack([
        xy[:, child] - xy[:, parent] for parent, child in BONE_EDGES
    ], axis=1)

    angles = []
    for first, center, last in ANGLE_TRIPLETS:
        first_vector = xy[:, first] - xy[:, center]
        last_vector = xy[:, last] - xy[:, center]
        denominator = (
            np.linalg.norm(first_vector, axis=1)
            * np.linalg.norm(last_vector, axis=1)
        )
        cosine = np.sum(first_vector * last_vector, axis=1) / np.maximum(
            denominator, 1e-6
        )
        angles.append(np.arccos(np.clip(cosine, -1.0, 1.0)) / np.pi)
    angles = np.stack(angles, axis=1).astype(np.float32)

    features = np.concatenate([
        xy.reshape(WINDOW, -1),
        velocity.reshape(WINDOW, -1),
        bones.reshape(WINDOW, -1),
        angles,
    ], axis=1).astype(np.float32)
    assert features.shape == (WINDOW, FEATURE_DIM), features.shape
    assert np.isfinite(features).all()
    return features


class PoseWindowDataset(Dataset):
    def __init__(self, indices, targets, training=False):
        self.indices = np.asarray(indices)
        self.targets = targets
        self.training = training

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, item):
        actual = self.indices[item]
        sample = X[actual].copy()
        if self.training:
            sample = augment_pose_window(sample)
        features = pose_to_features(sample).T
        return torch.from_numpy(features), torch.tensor(
            self.targets[actual], dtype=torch.long
        )


def make_loaders(targets, train_indices, val_indices, test_indices, train_weights, seed):
    generator = torch.Generator().manual_seed(seed)
    sampler = WeightedRandomSampler(
        torch.as_tensor(train_weights, dtype=torch.double),
        len(train_weights), replacement=True, generator=generator,
    )
    return (
        DataLoader(
            PoseWindowDataset(train_indices, targets, True),
            batch_size=BATCH_SIZE, sampler=sampler,
        ),
        DataLoader(
            PoseWindowDataset(val_indices, targets),
            batch_size=BATCH_SIZE, shuffle=False,
        ),
        DataLoader(
            PoseWindowDataset(test_indices, targets),
            batch_size=BATCH_SIZE, shuffle=False,
        ),
    )


gate_loaders = make_loaders(
    y_gate, train_idx, val_idx, test_idx, gate_train_weights, SEED
)
exercise_loaders = make_loaders(
    y_exercise, exercise_train_idx, exercise_val_idx, exercise_test_idx,
    exercise_train_weights, SEED + 1,
)
print("Feature blocks:", FEATURE_CONFIG)
print("Input features per frame:", FEATURE_DIM)
print("Gate split sizes:", [len(loader.dataset) for loader in gate_loaders])
print("Exercise split sizes:", [len(loader.dataset) for loader in exercise_loaders])


STREAMS = {"joint": 34, "bone": 24, "velocity": 34}
def pose_to_stream(sample, stream):
    xy = sample[:, :, :2].astype(np.float32)
    if stream == "joint": return xy.reshape(WINDOW, -1)
    if stream == "velocity":
        velocity = np.zeros_like(xy); velocity[1:] = xy[1:] - xy[:-1]
        return velocity.reshape(WINDOW, -1)
    if stream == "bone":
        return np.stack([xy[:, child] - xy[:, parent] for parent, child in BONE_EDGES], axis=1).reshape(WINDOW, -1)
    raise ValueError(stream)

class StreamDataset(Dataset):
    def __init__(self, indices, training, stream): self.indices=np.asarray(indices); self.training=training; self.stream=stream
    def __len__(self): return len(self.indices)
    def __getitem__(self,item):
        actual=self.indices[item]; sample=X[actual].copy()
        if self.training: sample=augment_pose_window(sample)
        return torch.from_numpy(pose_to_stream(sample,self.stream).T), torch.tensor(y_gate[actual],dtype=torch.long)

def make_stream_loaders(stream, seed):
    generator=torch.Generator().manual_seed(seed)
    sampler=WeightedRandomSampler(torch.as_tensor(gate_train_weights,dtype=torch.double),len(gate_train_weights),replacement=True,generator=generator)
    return (DataLoader(StreamDataset(train_idx,True,stream),batch_size=BATCH_SIZE,sampler=sampler),DataLoader(StreamDataset(val_idx,False,stream),batch_size=BATCH_SIZE,shuffle=False),DataLoader(StreamDataset(test_idx,False,stream),batch_size=BATCH_SIZE,shuffle=False))


## Visual audit of a phase-resampled cycle

The preview uses a real temporal candidate before and after the frozen V6-C
augmentation. It is exported with the report.


In [ ]:
SKELETON_EDGES = [
    (5, 6), (5, 7), (7, 9), (6, 8), (8, 10),
    (5, 11), (6, 12), (11, 12),
    (11, 13), (13, 15), (12, 14), (14, 16),
]
preview_candidates = train_idx[(labels[train_idx] == "squat") & temporal_mask[train_idx]]
preview_index = int(preview_candidates[0] if len(preview_candidates) else train_idx[0])
original_preview = X[preview_index].copy()
random_state = np.random.get_state()
np.random.seed(SEED + 1000)
augmented_preview = augment_pose_window(original_preview, force_all=True)
np.random.set_state(random_state)

fig, axes = plt.subplots(2, 6, figsize=(15, 6))
frame_indices = np.linspace(0, WINDOW - 1, 6, dtype=int)
for row, (title, sequence) in enumerate([
    ("original", original_preview), ("augmented", augmented_preview)
]):
    for axis, frame_index in zip(axes[row], frame_indices):
        xy = sequence[frame_index, :, :2]
        for left, right in SKELETON_EDGES:
            axis.plot(
                [xy[left, 0], xy[right, 0]],
                [-xy[left, 1], -xy[right, 1]],
                color="tab:blue", linewidth=1.4,
            )
        axis.scatter(xy[:, 0], -xy[:, 1], s=12, color="tab:red")
        axis.set_aspect("equal")
        axis.axis("off")
        axis.set_title(f"{title} t={frame_index}")
fig.suptitle(f"Notebook 26 phase-resampled cycle audit | {labels[preview_index]} | {window_keys[preview_index]}")
fig.tight_layout()
AUGMENTATION_PREVIEW_PATH = Path("/content/v6b_augmentation_preview.png")
fig.savefig(AUGMENTATION_PREVIEW_PATH, dpi=170, bbox_inches="tight")
plt.show()
print("Preview:", AUGMENTATION_PREVIEW_PATH)


## Three TCN gate streams and a frozen classifier

Joint coordinates, bone vectors, and velocity use independent TCN gates. The
temporal exercise classifier is never updated.


In [ ]:
class TemporalBlock(nn.Module):
    def __init__(self, in_channels, out_channels, dilation, dropout=0.2):
        super().__init__()
        padding = dilation
        self.net = nn.Sequential(
            nn.Conv1d(in_channels, out_channels, 3, padding=padding, dilation=dilation),
            nn.BatchNorm1d(out_channels), nn.ReLU(), nn.Dropout(dropout),
            nn.Conv1d(out_channels, out_channels, 3, padding=padding, dilation=dilation),
            nn.BatchNorm1d(out_channels), nn.ReLU(), nn.Dropout(dropout),
        )
        self.residual = (
            nn.Conv1d(in_channels, out_channels, 1)
            if in_channels != out_channels else nn.Identity()
        )

    def forward(self, x):
        return torch.relu(self.net(x) + self.residual(x))


class SmallTCN(nn.Module):
    def __init__(self, input_features, num_classes):
        super().__init__()
        channels = [64, 96, 128, 128]
        blocks, current = [], input_features
        for level, output in enumerate(channels):
            blocks.append(TemporalBlock(current, output, dilation=2 ** level))
            current = output
        self.features = nn.Sequential(*blocks)
        self.classifier = nn.Linear(current, num_classes)

    def forward(self, x):
        return self.classifier(self.features(x).mean(dim=-1))


checkpoint=torch.load(TEMPORAL_CHECKPOINT_PATH,map_location=DEVICE)
frozen_exercise_model=SmallTCN(FEATURE_DIM,len(SUPPORTED_CLASSES)).to(DEVICE)
frozen_exercise_model.load_state_dict(checkpoint["exercise_state_dict"])
frozen_exercise_model.eval()
for parameter in frozen_exercise_model.parameters(): parameter.requires_grad=False


In [ ]:
def train_gate(stream, loaders, seed):
    seed_everything(seed); model=SmallTCN(STREAMS[stream],2).to(DEVICE)
    criterion=nn.CrossEntropyLoss(); optimizer=torch.optim.AdamW(model.parameters(),lr=1e-3,weight_decay=1e-4)
    scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer,mode="max",patience=3,factor=0.5)
    best_path=Path(f"/content/best_v6f_{stream}.pt"); best_f1,stale,history=-1.0,0,[]
    def epoch(loader,training):
        model.train(training); losses,truth,pred=[],[],[]
        with (torch.enable_grad() if training else torch.no_grad()):
            for features,targets in loader:
                features,targets=features.to(DEVICE),targets.to(DEVICE)
                if training: optimizer.zero_grad(set_to_none=True)
                logits=model(features); loss=criterion(logits,targets)
                if training: loss.backward(); optimizer.step()
                losses.append(loss.item()*len(targets)); truth.extend(targets.cpu().numpy()); pred.extend(logits.argmax(1).cpu().numpy())
        return sum(losses)/len(truth),f1_score(truth,pred,average="macro",zero_division=0)
    for e in range(1,MAX_EPOCHS+1):
        tl,tf=epoch(loaders[0],True); vl,vf=epoch(loaders[1],False); scheduler.step(vf)
        history.append({"stream":stream,"epoch":e,"train_loss":tl,"train_macro_f1":tf,"val_loss":vl,"val_macro_f1":vf}); print(stream,e,round(tf,3),round(vf,3))
        if vf>best_f1+1e-4: best_f1,stale=vf,0; torch.save(model.state_dict(),best_path)
        else:
            stale+=1
            if stale>=PATIENCE: break
    model.load_state_dict(torch.load(best_path,map_location=DEVICE)); return model,history,best_f1

stream_results={}
for offset,stream in enumerate(STREAMS):
    loaders=make_stream_loaders(stream,SEED+100+offset)
    model,history,best=train_gate(stream,loaders,SEED+100+offset)
    stream_results[stream]={"model":model,"loaders":loaders,"history":history,"best_validation_f1":best}


## Validation-only CrossMax-inspired fusion

Validation compares mean probability, weakest-stream-weighted probability, and
the same score mixed with train-centroid distance. Selection requires at least
85% validation unknown rejection. Test labels are not used here.


In [ ]:
def infer_stream(model, indices, stream, batch_size=256):
    probabilities=[]; embeddings=[]; model.eval()
    with torch.no_grad():
        for start in range(0,len(indices),batch_size):
            batch=np.stack([pose_to_stream(X[i],stream) for i in indices[start:start+batch_size]]).transpose(0,2,1)
            tensor=torch.from_numpy(batch).to(DEVICE); hidden=model.features(tensor).mean(dim=-1)
            embeddings.append(hidden.cpu().numpy()); probabilities.append(model.classifier(hidden).softmax(1).cpu().numpy())
    return np.concatenate(probabilities),np.concatenate(embeddings)

def exercise_probabilities(indices):
    probabilities=[]
    with torch.no_grad():
        for start in range(0,len(indices),256):
            batch=np.stack([pose_to_features(X[i]) for i in indices[start:start+256]]).transpose(0,2,1)
            probabilities.append(frozen_exercise_model(torch.from_numpy(batch).to(DEVICE)).softmax(1).cpu().numpy())
    return np.concatenate(probabilities)

val_gate_prob={}; val_proto_prob={}; prototype_audit=[]
for stream,result in stream_results.items():
    train_prob,train_emb=infer_stream(result["model"],train_idx,stream)
    val_prob,val_emb=infer_stream(result["model"],val_idx,stream); val_gate_prob[stream]=val_prob[:,1]
    centroids={target:train_emb[y_gate[train_idx]==target].mean(axis=0) for target in [0,1]}
    scales={target:np.mean(np.sum((train_emb[y_gate[train_idx]==target]-centroids[target])**2,axis=1))+1e-6 for target in [0,1]}
    d0=np.sum((val_emb-centroids[0])**2,axis=1)/scales[0]; d1=np.sum((val_emb-centroids[1])**2,axis=1)/scales[1]
    val_proto_prob[stream]=np.exp(-d1)/(np.exp(-d0)+np.exp(-d1)+1e-8)
    result["centroids"],result["scales"]=centroids,scales
    prototype_audit.append({"stream":stream,"unknown_scale":scales[0],"supported_scale":scales[1]})

prob_matrix=np.stack([val_gate_prob[s] for s in STREAMS],axis=1)
proto_matrix=np.stack([val_proto_prob[s] for s in STREAMS],axis=1)
ordered=np.sort(prob_matrix,axis=1)
fusion_scores={"mean_probability":prob_matrix.mean(axis=1),"crossmax_probability":0.5*ordered[:,0]+0.3*ordered[:,1]+0.2*ordered[:,2],"crossmax_prototype":0.7*(0.5*ordered[:,0]+0.3*ordered[:,1]+0.2*ordered[:,2])+0.3*np.median(proto_matrix,axis=1)}
exercise_val=exercise_probabilities(val_idx).argmax(1)
selection_rows=[]
for fusion,score in fusion_scores.items():
    for threshold in np.linspace(0.05,0.95,37):
        pred=np.asarray([SUPPORTED_CLASSES[c] if p>=threshold else "unknown" for p,c in zip(score,exercise_val)])
        unknown=labels[val_idx]=="unknown"; supported=~unknown
        selection_rows.append({"fusion":fusion,"threshold":float(threshold),"validation_macro_f1":float(f1_score(labels[val_idx],pred,labels=ALL_CLASSES,average="macro",zero_division=0)),"supported_accuracy":float((pred[supported]==labels[val_idx][supported]).mean()),"unknown_rejection":float((pred[unknown]=="unknown").mean())})
selection=pd.DataFrame(selection_rows); eligible=selection[selection.unknown_rejection>=MIN_VALIDATION_UNKNOWN_REJECTION]
assert len(eligible),"No fusion satisfies the unknown-rejection constraint."
selected=eligible.sort_values(["validation_macro_f1","supported_accuracy"],ascending=False).iloc[0]
SELECTED_FUSION,SELECTED_THRESHOLD=selected.fusion,float(selected.threshold)
print("Selected",SELECTED_FUSION,SELECTED_THRESHOLD); display(selection.sort_values("validation_macro_f1",ascending=False).head(12))


## One selected gate on the unchanged test

Only the validation-selected fusion and threshold define the primary test
result. All stream scores and provenance are exported.


In [ ]:
test_gate_prob={}; test_proto_prob={}
for stream,result in stream_results.items():
    probability,embedding=infer_stream(result["model"],test_idx,stream); test_gate_prob[stream]=probability[:,1]
    c,s=result["centroids"],result["scales"]
    d0=np.sum((embedding-c[0])**2,axis=1)/s[0]; d1=np.sum((embedding-c[1])**2,axis=1)/s[1]
    test_proto_prob[stream]=np.exp(-d1)/(np.exp(-d0)+np.exp(-d1)+1e-8)
prob_matrix=np.stack([test_gate_prob[s] for s in STREAMS],axis=1); proto_matrix=np.stack([test_proto_prob[s] for s in STREAMS],axis=1); ordered=np.sort(prob_matrix,axis=1)
test_fusions={"mean_probability":prob_matrix.mean(axis=1),"crossmax_probability":0.5*ordered[:,0]+0.3*ordered[:,1]+0.2*ordered[:,2],"crossmax_prototype":0.7*(0.5*ordered[:,0]+0.3*ordered[:,1]+0.2*ordered[:,2])+0.3*np.median(proto_matrix,axis=1)}
score=test_fusions[SELECTED_FUSION]; exercise_prob=exercise_probabilities(test_idx); exercise_pred=exercise_prob.argmax(1)
end_pred=np.asarray([SUPPORTED_CLASSES[c] if p>=SELECTED_THRESHOLD else "unknown" for p,c in zip(score,exercise_pred)])
gate_pred=(score>=SELECTED_THRESHOLD).astype(int); supported=labels[test_idx]!="unknown"; classifier_truth=labels[test_idx][supported]; classifier_pred=np.asarray([SUPPORTED_CLASSES[c] for c in exercise_pred[supported]])
metrics={"experiment":"v6f_multistream_open_set_gate","pinned_source_commit":PINNED_COMMIT,"selected_on_validation":{"fusion":SELECTED_FUSION,"threshold":SELECTED_THRESHOLD,"minimum_unknown_rejection":MIN_VALIDATION_UNKNOWN_REJECTION,"row":selected.to_dict()},"stream_best_validation_f1":{s:r["best_validation_f1"] for s,r in stream_results.items()},"gate":{"accuracy":float(accuracy_score(y_gate[test_idx],gate_pred)),"balanced_accuracy":float(balanced_accuracy_score(y_gate[test_idx],gate_pred)),"macro_f1":float(f1_score(y_gate[test_idx],gate_pred,average="macro",zero_division=0))},"exercise_classifier_oracle":{"accuracy":float(accuracy_score(classifier_truth,classifier_pred)),"balanced_accuracy":float(balanced_accuracy_score(classifier_truth,classifier_pred)),"macro_f1":float(f1_score(classifier_truth,classifier_pred,labels=SUPPORTED_CLASSES,average="macro",zero_division=0))},"end_to_end":{"accuracy":float(accuracy_score(labels[test_idx],end_pred)),"balanced_accuracy":float(balanced_accuracy_score(labels[test_idx],end_pred)),"macro_f1":float(f1_score(labels[test_idx],end_pred,labels=ALL_CLASSES,average="macro",zero_division=0))},"controls":{"original_v6c":baseline_metrics,"notebook24_data_only":data_only_metrics,"notebook26_temporal":temporal_metrics}}
REPORT_ROOT=Path("/content/calorie_v6f_multistream_open_set_gate_report")
if REPORT_ROOT.exists(): shutil.rmtree(REPORT_ROOT)
REPORT_ROOT.mkdir(); (REPORT_ROOT/"metrics.json").write_text(json.dumps(metrics,indent=2)+"\n")
selection.to_csv(REPORT_ROOT/"validation_fusion_threshold_sweep.csv",index=False); pd.DataFrame(prototype_audit).to_csv(REPORT_ROOT/"prototype_audit.csv",index=False); pd.DataFrame(sum([r["history"] for r in stream_results.values()],[])).to_csv(REPORT_ROOT/"training_history.csv",index=False)
pd.DataFrame({"window_key":window_keys[test_idx],"group_id":groups[test_idx],"true_class":labels[test_idx],**{f"{s}_supported_probability":test_gate_prob[s] for s in STREAMS},"prototype_supported_probability":np.median(proto_matrix,axis=1),"fused_supported_score":score,"exercise_prediction":[SUPPORTED_CLASSES[c] for c in exercise_pred],"predicted_class":end_pred}).to_csv(REPORT_ROOT/"test_window_predictions.csv",index=False)
original_by_key=dict(zip(baseline_predictions.window_key,baseline_predictions.predicted_class)); temporal_by_key=dict(zip(temporal_predictions.window_key,temporal_predictions.predicted_class)); creator_rows=[]
for group_id in sorted(set(groups[test_idx])):
    mask=groups[test_idx]==group_id; truth=labels[test_idx][mask]; prediction=end_pred[mask]; present=sorted(set(truth)); original=np.asarray([original_by_key[k] for k in window_keys[test_idx][mask]]); temporal=np.asarray([temporal_by_key[k] for k in window_keys[test_idx][mask]])
    current_f1=f1_score(truth,prediction,labels=present,average="macro",zero_division=0)
    creator_rows.append({"group_id":group_id,"windows":int(mask.sum()),"current_accuracy":float(accuracy_score(truth,prediction)),"current_present_macro_f1":float(current_f1),"delta_accuracy_vs_original":float(accuracy_score(truth,prediction)-accuracy_score(truth,original)),"delta_present_f1_vs_original":float(current_f1-f1_score(truth,original,labels=present,average="macro",zero_division=0)),"delta_accuracy_vs_temporal":float(accuracy_score(truth,prediction)-accuracy_score(truth,temporal)),"delta_present_f1_vs_temporal":float(current_f1-f1_score(truth,temporal,labels=present,average="macro",zero_division=0))})
pd.DataFrame(creator_rows).to_csv(REPORT_ROOT/"test_creator_breakdown.csv",index=False)
sampler_audit.to_csv(REPORT_ROOT/"sampler_audit.csv",index=False); temporal_frame.to_csv(REPORT_ROOT/"temporal_cycle_audit.csv",index=False); data_delta_audit.to_csv(REPORT_ROOT/"data_delta_audit.csv",index=False); class_creator_support_matrix.to_csv(REPORT_ROOT/"class_creator_support_matrix.csv")
pd.DataFrame({"window_key":window_keys,"group_id":groups,"active_class":labels,"split":assigned_split}).query('split != ""').to_csv(REPORT_ROOT/"split_assignments.csv",index=False)
for name,truth,pred,lab,ticks in [("gate",y_gate[test_idx],gate_pred,[0,1],["unknown","supported"]),("exercise",classifier_truth,classifier_pred,SUPPORTED_CLASSES,SUPPORTED_CLASSES),("end_to_end",labels[test_idx],end_pred,ALL_CLASSES,ALL_CLASSES)]:
    (REPORT_ROOT/f"{name}_classification_report.json").write_text(json.dumps(classification_report(truth,pred,labels=lab,target_names=ticks if name=="gate" else None,output_dict=True,zero_division=0),indent=2)+"\n")
    matrix=confusion_matrix(truth,pred,labels=lab); plt.figure(figsize=(max(6,len(ticks)*.75),max(5,len(ticks)*.65))); sns.heatmap(matrix,annot=True,fmt="d",cmap="Blues",xticklabels=ticks,yticklabels=ticks); plt.tight_layout(); plt.savefig(REPORT_ROOT/f"{name}_confusion_matrix.png",dpi=180); plt.close()
for stream,result in stream_results.items(): torch.save({"state_dict":result["model"].state_dict(),"stream":stream,"input_features":STREAMS[stream],"centroids":result["centroids"],"scales":result["scales"]},REPORT_ROOT/f"{stream}_gate.pt")
shutil.copy2(TEMPORAL_CHECKPOINT_PATH,REPORT_ROOT/"frozen_temporal_classifier_checkpoint.pt")
source_paths=zip_paths+[HARD_POSITIVE_PATH,V6_SPLIT_PATH,BASELINE_METRICS_PATH,BASELINE_PREDICTIONS_PATH,DATA_ONLY_METRICS_PATH,DATA_ONLY_PREDICTIONS_PATH,TEMPORAL_CHECKPOINT_PATH,TEMPORAL_METRICS_PATH,TEMPORAL_PREDICTIONS_PATH]
(REPORT_ROOT/"source_hashes.json").write_text(json.dumps({p.name:sha256(p) for p in source_paths},indent=2)+"\n")
lines=[f"{sha256(p)}  {p.relative_to(REPORT_ROOT)}" for p in sorted(x for x in REPORT_ROOT.rglob('*') if x.is_file())]; (REPORT_ROOT/"SHA256SUMS.txt").write_text("\n".join(lines)+"\n")
zip_path=shutil.make_archive("/content/calorie_v6f_multistream_open_set_gate_report","zip",root_dir=REPORT_ROOT); print(json.dumps(metrics,indent=2)); files.download(zip_path)


## Return the report

Send back `calorie_v6f_multistream_open_set_gate_report.zip` before
self-supervised pretraining is prepared.
